# 03 - Purpose/Risk extraction: run, score, compare

A place to *look at things*, not a place where long jobs live. Every cell
imports and calls the pipeline -- no stage is reimplemented here, and nothing
shells out to a subprocess.

A 14-day corpus run does **not** belong in a notebook: a kernel will not
survive it, and this repo has already had `.ipynb` edits silently clobbered by
a stale editor tab. For that, use the CLI under `nohup`/`tmux`:

```
python -m engine.pipeline.cli run --records <pilot.jsonl> --workers 1 \
    --note "full corpus" > run.log 2>&1 &
```

Then come back here and read the run directory it wrote.

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from pathlib import Path

from engine.pipeline import run as run_mod, store
from engine.pipeline.compare import compare
from engine.pipeline.config import Config, Decode
from engine.pipeline.score import score

RECORDS = "data/dsld/runs/20260908T125504Z/raw/pilot-1000.jsonl"
GOLD = "datasets/pilot/gold-100.jsonl"

gold = [json.loads(l) for l in Path(GOLD).read_text().splitlines() if l.strip()]
len(gold)

## The config, and what its hash covers

Two runs pool only when their `config_hash` matches. The hash covers prompt,
model, decode params, schema and renderer version -- and `enable_thinking`
explicitly, because this model ships with thinking on and a silent flip would
make every result incomparable. The git commit is deliberately *not* hashed,
so two identical runs a week apart still pool.

In [ ]:
cfg = Config()
print("config_hash :", cfg.config_hash)
print("prompt_sha  :", cfg.prompt_sha[:16])
print("prompt file :", cfg.prompt_path)
cfg.identity

## Smoke: is the inference path honest?

Five gold labels through the **real** model, prompt and grammar, asserting
that evidence spans come back verbatim. This is the corruption alarm: this
architecture has a known CUDA bug that makes the model load normally and emit
fluent garbage, and a 2-token "ok" generation cannot tell that apart from a
correct model. Reproducing exact substrings from a 1,600-4,000 char passage
can. `run()` calls this before every batch.

In [ ]:
run_mod.smoke(cfg, records_path=RECORDS)

## A small batch

`run()` is a plain function returning the run directory -- the CLI calls this
same function, so behaviour cannot diverge between the two callers. Progress
prints as plain appended lines (never ANSI redraw, which fills a notebook cell
with thousands of garbage lines).

In [ ]:
run_dir = run_mod.run(
    RECORDS,
    config=cfg,
    limit=20,
    n_workers=1,
    note="baseline: prompt as the gold set was built with",
)
run_dir

## What the run produced

In [ ]:
run_mod.stats(run_dir)

Rows are plain JSONL and carry `config_hash` and `provenance` inline, so a
row is interpretable without joining it back to a manifest.

In [ ]:
rows = store.read_rows(run_dir)
rows[0]

## Score it against the gold set

Headline is per-field presence/absence over 2 cells per label, with
hallucination rate called out separately. The detail number is evidence-span
precision/recall per claim; its ceiling is documented in the scorer's
docstring (29% of gold risk entries share an evidence block with a sibling, so
span matching cannot separate them -- a metric limit, never model error).

The gold set is **one frontier model's reading** of the label, not a human
adjudication, so these numbers mean *agreement with that reading*.

In [ ]:
metrics = score(rows, gold)
metrics

## Iterate: edit the prompt, run again, read the diff above the delta

Edit `prompts/purpose_risk.md`, rerun the batch with a `--note` saying what you
were trying to achieve, then compare. The prompt diff prints above the metric
delta because the diff is the thing actually worth reading -- a sha only says
*that* something changed.

In [ ]:
# after editing prompts/purpose_risk.md and running a second batch:
# print(compare(run_dir_a, run_dir_b, score(store.read_rows(run_dir_a), gold),
#                                     score(store.read_rows(run_dir_b), gold)))

## Retrying what broke

`empty` is a success and is never retried. A retry pass covers `error`,
`unparseable` and `poison` rows only, and it writes its **own** run directory
so retried work stays attributable rather than blended into the original --
whose failure rate stays readable as the early signal that a prompt or grammar
is wrong.

In [ ]:
run_mod.retry_ids(run_dir)   # what a retry would cover
# run_mod.retry(run_dir, RECORDS, config=cfg)

## Comparing many runs

Runs load into pandas by globbing run directories -- no database needed.

In [ ]:
df = store.load_runs()
df.groupby(["run_id", "status"]).size() if len(df) else df